# 🌐 NetOps-SLM: Mixture of LoRA Experts (MoLE) Multi-Adapter Training
### **Continental-Scale 4-Node Spine-Leaf CLOS Telemetry & Protocol State Machine Specialization**

**Hardware Target:** Free Google Colab (1x NVIDIA Tesla T4, 15GB VRAM) or Kaggle (2x T4)
**Base Model:** `Qwen/Qwen2.5-Coder-1.5B-Instruct` (4-bit NF4 Quantization, ~2.2 GB VRAM peak)
**Experts Trained:**
1. `expert_bgp_protocol`: 73 authentic single-router BGP FSM episodes
2. `expert_clos_fabric`: 5 authentic 4-switch Spine-Leaf CLOS multi-path episodes
3. `expert_template_gate`: Strict schema synthesis and TypedActionGate templates

**Total Cloud Spend:** **$0.00**

## 1. Hardware & Environment Audit
Verify GPU availability (NVIDIA Tesla T4 with 15GB VRAM).

In [ ]:
!nvidia-smi

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

## 2. Install ML & Quantization Dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets trl

## 3. Extract MoLE Datasets & Pipeline (Zero-Setup Auto-Extraction)
This cell automatically extracts all 78 authentic Containerlab lab episodes, MoLE architecture code, and the training runner.
*(No git credentials or manual file uploads required)*

In [ ]:
import os, io, base64, zipfile

# Option 1: If NetOps_MoLE_Colab_Bundle.zip was uploaded to session storage, extract it
if os.path.exists("NetOps_MoLE_Colab_Bundle.zip"):
    print("[+] Found uploaded NetOps_MoLE_Colab_Bundle.zip. Extracting...")
    with zipfile.ZipFile("NetOps_MoLE_Colab_Bundle.zip", "r") as zf:
        zf.extractall(".")
else:
    # Option 2: Auto-extract from embedded offline archive
    print("[*] Auto-extracting embedded 78-episode MoLE dataset and source files...")
    EMBEDDED_ZIP_B64 = "..."
    raw_zip = base64.b64decode(EMBEDDED_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(raw_zip)) as zf:
        zf.extractall(".")

print("[+] Extraction complete! Available directories:", [d for d in os.listdir(".") if os.path.isdir(d)])

## 4. Verify Partitioned MoLE Datasets
Audits the 78 authentic Containerlab lab episodes across the 3 expert partitions.

In [ ]:
import os

for f in ["mole_expert_bgp.jsonl", "mole_expert_clos.jsonl", "mole_expert_template.jsonl"]:
    p = os.path.join("dataset", f)
    if os.path.exists(p):
        with open(p, "r", encoding="utf-8") as fp:
            lines = len(fp.readlines())
        print(f"[+] {f:<30}: {lines} authentic examples")
    else:
        print(f"[X] Missing: {p}")

## 5. Train All 3 MoLE Experts (Sequential 4-bit QLoRA Execution)
Runs the unified Colab training runner to fine-tune:
- `expert_bgp_protocol` (Rank 16, Attention projections)
- `expert_clos_fabric` (Rank 16, All-linear projections including MLP gates)
- `expert_template_gate` (Rank 8, Syntax & Schema)

In [ ]:
!python notebooks/train_mole_colab.py

## 6. Validate Trained Adapters with MoLE Gating Network & Digital Twin

In [ ]:
import sys
sys.path.insert(0, ".")

from src.models.mole_router import MoLERouter
from src.verification.digital_twin_verifier import DigitalTwinVerifier

router = MoLERouter()
verifier = DigitalTwinVerifier()

test_desc = "Device spine-1 has asymmetric MTU packet drops on peer 10.77.0.1. Generate safe typed drain action."
decision = router.route(test_desc)
print(f"Selected Expert: {decision.selected_expert}")
print(f"Expert Weights:  {decision.weights}")
print(f"Dispatch Time:   {decision.reasoning}")

proof = verifier.verify_remediation_preflight(
    incident_id="EPISODE_CLOS_003_MTU_MISMATCH",
    action_payload={
        "action": "drain_spine_traffic",
        "parameters": {"device": "spine-1", "neighbor": "10.77.0.1"}
    }
)
print(f"Digital Twin Pre-Flight Status: {'PASSED [OK]' if proof.preflight_passed else 'FAILED'}")
print(f"Forwarding Paths: {proof.active_forwarding_paths}")
print(f"Cryptographic SHA-256 Signature: {proof.cryptographic_sha256}")

## 7. Download Trained MoLE Adapters (`mole_trained_adapters.zip`)
Bundles all 3 adapter directories for immediate integration back into your local IDE.

In [ ]:
import os
from google.colab import files

zip_path = "mole_trained_adapters.zip"
if os.path.exists(zip_path):
    print(f"[+] Downloading {zip_path} ({os.path.getsize(zip_path)/(1024**2):.2f} MB)...")
    files.download(zip_path)
else:
    print(f"[X] {zip_path} not found. Ensure training completed in Step 5.")